# FST-Arch: run all experiments

This notebook reproduces the simulation section of the article **Growing and pruning neural networks by fixed-sequence testing with family-wise error control**.

Steps:
1. Get the code.
2. Compute the Monte Carlo *true* risks $\mu(a)$ of every architecture (`oracle.py`).
3. Choose the margins $\gamma$ (growing) and $\delta$ (pruning) and check which null hypotheses are true.
4. Run the replications (`sim.py`) and print Tables 1 and 2.
5. Plot the results.

Only `numpy`, `scipy` and `matplotlib` are needed; all are preinstalled on Colab. No GPU is needed.

## 1. Get the code

In [ ]:
import os, sys
if not os.path.exists('fst-architecture-selection'):
    !git clone -q https://github.com/yamenetoo/fst-architecture-selection.git
%cd fst-architecture-selection/experiments
!pip -q install -r ../requirements.txt
sys.path.insert(0, os.getcwd())

## 2. True architecture risks

For one fixed training sample, $\mu(a)$ is estimated with 300 seeds and 40 000 test points for every width $\{4,\dots,256\}$ and depth $\{1,2,3\}$, and for every pruning level of the width-256 network. This is the slowest step (a few minutes on Colab).

In [ ]:
import time
t0 = time.time()
!python oracle.py
print(f'oracle time: {time.time() - t0:.0f} s')

## 3. Choose the margins

$\gamma$ is the smallest improvement worth growing for and $\delta$ the largest loss accepted when pruning, both in units of the loss (MSE). Pick them so that the width chain has **both** true and false null hypotheses; otherwise the FWER column is trivially 0. The cell below prints the truth table for your choice.

In [ ]:
B = 1000        # Monte Carlo replications (use 100 for a quick test)
GAMMA = 0.005   # growing margin
DELTA = 0.01    # pruning margin

import numpy as np
from oracle import WIDTHS, DEPTHS, FRACS
orc = np.load('oracle.npz')
mu, mu_pr = orc['risk'].mean(0), orc['prune'].mean(0)

print('Growing, width chain at depth 1: H0 true if mu(w+) - mu(w) >= -gamma')
for i in range(len(WIDTHS) - 1):
    th = mu[i + 1, 0] - mu[i, 0]
    print(f'  {WIDTHS[i]:>3} -> {WIDTHS[i+1]:>3}: theta = {th:+.4f}   H0 {"TRUE" if th >= -GAMMA else "false"}')
print('\nPruning from width 256: H0 true if mu(a_k) - mu(a_0) >= delta')
for k, fr in enumerate(FRACS[1:], 1):
    h = mu_pr[k] - mu_pr[0]
    print(f'  remove {fr:.0%}: loss = {h:+.4f}   H0 {"TRUE" if h >= DELTA else "false"}')

## 4. Run the replications

Each replication draws new validation splits and seeds and runs the growing procedure with five standard errors and the pruning procedure with two margins.

In [ ]:
t0 = time.time()
!python sim.py {B} {GAMMA} {DELTA}
print(f'simulation time: {time.time() - t0:.0f} s')

In [ ]:
print('Table 1 rows (growing) -- paste into the article:\n')
print(open('table_building.tex').read())
print('Table 2 rows (pruning):\n')
print(open('table_pruning.tex').read())

## 5. Plots

In [ ]:
import json
import matplotlib.pyplot as plt
res = json.load(open('results.json'))

names = ['crossed', 'crossed+safety', 'seed_only', 'single', 'welch']
labels = ['Crossed\n(proposed)', 'Crossed\n+ safety', 'Seed-only', 'Single\nseed', 'Welch']
fwer = [res['build'][m]['fwer'] for m in names]
se = [res['build'][m]['fwer_se'] for m in names]
excess = [res['build'][m]['excess'] for m in names]

fig, ax = plt.subplots(1, 3, figsize=(15, 4))
ax[0].bar(labels, fwer, yerr=[1.96 * s for s in se], color='#4c72b0')
ax[0].axhline(0.05, ls='--', color='k', lw=1, label='nominal 0.05')
ax[0].set_title('Growing: family-wise error rate'); ax[0].legend()

ax[1].bar(labels, excess, color='#55a868')
ax[1].set_title('Growing: mean excess risk of selected network')

pk = ['cumulative', 'stepwise']
pe = [res['prune'][k]['harm_exceeds'] for k in pk]
ps = [res['prune'][k]['harm_exceeds_se'] for k in pk]
ax[2].bar(['Cumulative\n(proposed)', 'Stepwise'], pe, yerr=[1.96 * s for s in ps], color=['#4c72b0', '#c44e52'])
ax[2].axhline(0.05, ls='--', color='k', lw=1)
ax[2].set_title(r'Pruning: $P\{\mu(\hat a)-\mu(a_0)\geq\delta\}$')
for a in ax: a.tick_params(axis='x', labelsize=9)
plt.tight_layout(); plt.savefig('results.png', dpi=200); plt.show()

if res.get('calib'):
    print(f"Size of a single test at the true null closest to the boundary ({res['calib_pair']}), nominal 0.025:")
    for m, v in res['calib'].items():
        print(f'  {m:<10} {v:.3f}')

## 6. Download the results

`results.json`, `table_building.tex`, `table_pruning.tex`, `oracle.npz` and `results.png` are in the `experiments` folder (Files panel on the left).

In [ ]:
try:
    from google.colab import files
    for f in ['results.json', 'table_building.tex', 'table_pruning.tex', 'results.png']:
        files.download(f)
except ImportError:
    print('Not on Colab: files are in', os.getcwd())